# Day 01 — 为什么需要分布式训练，DDP 如何解决它

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/day-01-distributed-ddp.ipynb)

- **日期**：2026-09-30（2026-10-02 重写：补全前因后果）
- **定位**：并行基础线的第一篇。目标不是背 API，而是能回答一条完整的因果链：单卡为什么训不动 → 数据并行怎么切 → 梯度同步为什么贵 → ring all-reduce 为什么省 → DDP 还差什么。
- **环境**：代码用 CPU + `gloo` 后端验证机制，Colab 免费版可跑；GPU/NCCL 的量级差异在文中标注。

记号： $ P $ = 参数量（个）， $ N $ = 进程数（rank 数 = GPU 数）， $ S $ = 一个 tensor 的字节数， $ B $ = 单链路带宽（bytes/s）， $ \alpha $ = 一次点对点消息的固定延迟（s）。


## 0. 这篇的因果链（先看地图）

1. 训练大模型时，单卡撞上**两堵墙**：显存装不下、时间耗不起。
2. **数据并行**是最直接的拆法：模型复制 $ N $ 份，batch 切成 $ N $ 份，各算各的。
3. 但复制带来新问题：$ N $ 份参数必须步调一致 → 每一步都要**同步梯度**，而同步的通信量和参数量同阶。
4. 同步怎么做？参数服务器死于单点带宽；**all-reduce** 让每个 rank 既生产又消费，没有中心瓶颈。
5. all-reduce 的最优拓扑是**环**：每条链路负载均衡，总通信量 $ 2(N-1)/N \cdot S $ ，$ N $ 大时趋近与 $ N $ 无关的 $ 2S $ 。
6. 但小梯度 tensor 太多时**延迟项**主导 → DDP 把梯度**分桶**（bucketing）攒大再发，并让通信与反向计算**重叠**。
7. DDP 解决了同步的效率，却没动显存墙：每份副本仍是全量参数 + 全量优化器状态 → 这是下一篇 ZeRO/FSDP 要解决的问题。

下面逐环展开，每环都算账。


## 1. 第一问：单卡为什么训不动 —— 两堵墙

**显存墙。** 训练要存的不只是参数。以混合精度 + Adam 为例，每个参数需要：

| 内容 | 精度 | 字节 |
|---|---|---|
| 参数（计算用） | bf16 | 2 |
| 梯度 | bf16 | 2 |
| 参数（优化器主副本） | fp32 | 4 |
| Adam 一阶动量 $ m $ | fp32 | 4 |
| Adam 二阶动量 $ v $ | fp32 | 4 |
| **合计** | | **16 字节/参数** |

70B 参数 → $ 70\times10^9 \times 16 \approx 1.1\text{TB} $ 。一张 H100 是 80GB。**光参数+优化器状态就要 14 张卡**，还没算激活值。显存墙是硬约束：装不下就是装不下，写代码技巧绕不过去。

**时间墙。** 训练的计算量约 $ 6P $ FLOPs/token（前向 $ 2P $ 、反向 $ 4P $ ）。10T token、70B 模型 → $ 6 \times 7\times10^{10} \times 10^{13} = 4.2\times10^{24} $ FLOPs。单卡 H100 有效算力约 $ 5\times10^{14} $ FLOPS → 需要 $ 8.4\times10^9 $ 秒 ≈ **266 年**。

结论：并行不是优化，是生存条件。接下来的问题只是：怎么并行、代价是什么。


## 2. 数据并行：把 batch 切开

最直接的拆法：$ N $ 张卡各放一份**完整模型**，把一个大 batch 切成 $ N $ 份，每份在一个 rank 上独立做前向+反向。

为什么数学上成立？大 batch 的梯度是各样本梯度的平均，平均可以分片算再合并：

$$ \nabla \mathcal{L}_{batch} = \frac{1}{N} \sum_{i=1}^{N} \nabla \mathcal{L}_{分片 i} $$

所以只要每步结束时把 $ N $ 份梯度**求平均**再更新，数据并行与单卡大 batch 严格等价（只差浮点误差）。

代价也在这里诞生了：梯度和参数同尺寸（fp32 下 $ 4P $ 字节，bf16 下 $ 2P $ ）。70B 模型每步要同步 140–280GB 量级的数据。**同步的效率直接决定数据并行能不能扩展** —— 这是后面所有内容的动机。


## 3. 同步怎么做：参数服务器为什么死

**方案 A：参数服务器（parameter server）。** 每个 worker 算完梯度发给中心 server，server 平均后把新参数广播回来。

账算一下：server 的网卡每步要收 $ N $ 份梯度、发 $ N $ 份参数，流量 $ 2NS $ （ $ S $ = 参数字节数）。 $ N=1024 $ 、 $ S=140 $ GB 时是 287TB 流过一块网卡 —— 卡数越多，中心越堵，**扩展性死在单点带宽上**。这是结构性问题，换更快的 server 网卡只是推迟。

**方案 B：all-reduce。** 没有中心。每个 rank 既贡献自己那份梯度、也收齐所有人的份数，自己在本地平均。负载均匀摊到所有链路上 —— 这是对单点瓶颈的结构性回应。

all-reduce 只是语义（人人拿到规约结果），实现可以用多种拓扑。哪种拓扑通信量最小？这就是 ring 的入场理由。


## 4. Ring all-reduce：为什么是环

把大小 $ S $ 的 tensor 切成 $ N $ 块（每块 $ S/N $ ）， $ N $ 个 rank 排成一个环，分两步：

- **scatter-reduce**： $ N-1 $ 轮。每轮每个 rank 把自己的一块发给右邻、从左邻收一块并累加。轮转 $ N-1 $ 次后，每个 rank 恰好持有一块**完整求和**的结果（每人负责不同块）。
- **all-gather**：再 $ N-1 $ 轮，把各自手里那块完整结果在环上传一圈，人人集齐 $ N $ 块。

每个 rank 发送量：每轮发 $ S/N $ ，共 $ 2(N-1) $ 轮：

$$ \text{每 rank 发送量} = 2(N-1)\cdot\frac{S}{N} = \frac{2(N-1)}{N}S $$

时间模型（延迟 $ \alpha $ + 带宽两项）：

$$ t \approx 2(N-1)\alpha + \frac{2(N-1)}{N}\cdot\frac{S}{B} $$

**为什么环是最优的？** 注意每条链路在每一轮都在满载传一块，没有空转、没有热点 —— 任何拓扑里每块数据都至少要离开本地、结果至少要回来，下界就是 $ \approx 2S $ ；环达到了这个下界（ $ N\to\infty $ 时 $ 2(N-1)/N \to 2 $ ）。对比参数服务器的 $ 2NS $ （server 视角），环把压力从 $ O(N) $ 降到 $ O(1) $ 。这就是"带宽最优"的确切含义。

代价在延迟项： $ 2(N-1)\alpha $ 随 $ N $ 线性涨。 $ S $ 大时带宽项主导、无所谓； $ S $ 小时延迟说了算 —— 记住这个伏笔，第 6 节回收。


In [ ]:
def ring_allreduce_time(S_GB, N, B_GBs, alpha_us=20):
    """ring all-reduce 的时延估计。

    S_GB: 要规约的数据量（GB）；N: rank 数；B_GBs: 链路带宽（GB/s）；
    alpha_us: 单跳时延（微秒）。
    返回: 总时间（毫秒）= 2(N−1)·α + 2(N−1)/N · S/B。"""
    S, B = S_GB * 1e9, B_GBs * 1e9
    t = 2 * (N - 1) * alpha_us * 1e-6 + 2 * (N - 1) / N * S / B
    return t * 1e3

print("1GB tensor 的 all-reduce：")
for N, B, link in [(8, 50, "NVLink"), (64, 25, "IB 200G"), (1024, 25, "IB 200G")]:
    t = ring_allreduce_time(1.0, N, B)
    ps = 2 * N * 1.0 / B * 1e3  # 参数服务器的 server 侧流量时间（收发 2NS）
    print(f"  N={N:4d} [{link:8s}] ring={t:6.1f} ms | 参数服务器中心节点={ps:8.1f} ms")
print("\n读法：N 从 8 涨到 1024，ring 几乎不变（带宽最优），参数服务器线性爆炸。")


## 5. 回收伏笔：为什么 DDP 要分桶（bucketing）

真实模型的梯度不是一块大 tensor，而是几百个形状各异的参数梯度（每层一个）。如果每个梯度 backward 一出来就单独 all-reduce 一次：

- 每次通信量 $ S_i $ 很小（KB 级），延迟项 $ 2(N-1)\alpha $ 完胜带宽项 → 每步几百次通信，时间几乎全花在延迟上。
- 且小消息无法打满链路带宽。

DDP 的解法：按参数在模型中的**逆序**（≈ backward 产生梯度的顺序）把梯度拼进固定大小的桶（默认 25MB），桶满立刻发起一次 all-reduce。几百次小通信变几次大通信，延迟项从 $ 500 \times 2(N-1)\alpha $ 降到个位数次。

第二个优化顺手就来了：**通信与计算重叠**。桶在 backward 中途就满、就发，所以靠前层的梯度通信与靠后层的 backward 计算同时进行。理想情况下单步时间 $ \approx \max(t_{compute}, t_{comm}) $ 而非两者之和 —— DDP 扩展效率高的核心原因就是这个重叠，而不只是 ring 本身。


## 6. DDP 全貌与它的两个一致性前提

把前面拼起来，DDP 每步做的是：

1. 各 rank 用自己那份数据分片 forward + backward；
2. backward 过程中梯度按桶 all-reduce（求和），除以 $ N $ 后写回 `.grad`；
3. 各 rank 用**相同**的平均梯度更新参数。

由此推出两个必须成立的前提，调试清单里的坑全从这里来：

- **初始参数必须 bit 级一致**：否则平均的是不同起点的梯度，更新后仍然不一致、越走越散。对策：各 rank 同种子初始化（更稳妥的是 rank 0 广播一次）。
- **每个 rank 每步的 collective 次数必须相同**：all-reduce 是集合通信，少一个 rank 进来就全体挂起（hang）。分支逻辑导致某 rank 少走一次 backward，就会死锁。


**运行方式（这一节和后面所有 spawn 演示通用）**：代码先用 `%%writefile` 存成 `.py` 脚本、再以脚本方式运行，不能把函数直接定义在 cell 里 `mp.spawn` —— spawn 的子进程要重新 import `__main__` 才能拿到入口函数，而 notebook 的 `__main__` 是 kernel 本人，子进程拿不到你定义的函数，会直接 `ProcessExitedException` 退出。落成脚本后 `__main__` 就是脚本本人，子进程正常 import；这也正是生产里 `torchrun` 拉起脚本的形态。

In [ ]:
%%writefile ddp_smoke.py
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

def run(rank, world_size):
    """mp.spawn 的入口，每个进程各跑一份本函数。

    rank: 本进程编号 [0, world_size)（真实训练里对应一张卡/一个节点角色）；
    world_size: 进程总数，本 notebook 固定 2（模拟 2 卡）。"""
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29501'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)

    torch.manual_seed(0)  # 前提 1：初始参数一致
    model = DDP(torch.nn.Linear(10, 1))
    opt = torch.optim.SGD(model.parameters(), lr=0.1)

    x = torch.randn(32, 10); y = torch.randn(32, 1)
    opt.zero_grad()
    loss = ((model(x) - y) ** 2).mean()
    loss.backward()   # DDP 在 autograd hook 里做分桶 all-reduce
    opt.step()

    # 验证前提的推论：更新后各 rank 参数应 bit 级一致
    params = torch.cat([p.data.view(-1) for p in model.parameters()])
    gathered = [torch.zeros_like(params) for _ in range(world_size)]
    dist.all_gather(gathered, params)
    if rank == 0:
        for r in range(1, world_size):
            d = (gathered[0] - gathered[r]).abs().max().item()
            print(f'rank0 vs rank{r}: 参数最大差 = {d:.2e} （应为 0）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run, args=(2,), nprocs=2, join=True)


In [ ]:
!python3 ddp_smoke.py

## 7. 实验：实测通信成本曲线，拟合 $ \alpha $–$ \beta $ 模型

理论公式 $ t \approx 2(N-1)\alpha + \frac{2(N-1)}{N}\cdot\frac{S}{B} $ 里， $ \alpha $ 和 $ B $ 是机器相关的。用实测拟合出这两个数，以后估算任何并行方案的通信开销都靠它（面试里"1024 卡 all-reduce 280GB 要多久"这类问题就是这么答的）。


In [ ]:
%%writefile ddp_bench.py
import os, time, torch
import torch.distributed as dist
import torch.multiprocessing as mp

def bench(rank, world_size):
    """mp.spawn 入口（rank / world_size 含义同 run）：实测 all-reduce 的时延-带宽曲线。

    对 4 个消息尺寸各跑 20 次 all-reduce 计时，rank 0 再把 t 与消息大小 S 做线性拟合，
    分解出单跳时延 α（μs/op）和有效带宽（GB/s）——和 ring_allreduce_time 的公式互相对照。"""
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29502'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    N = world_size
    sizes = [4*1024, 64*1024, 1024*1024, 16*1024*1024]
    ts = []
    for S in sizes:
        t = torch.zeros(S // 4)
        for _ in range(5): dist.all_reduce(t)
        dist.barrier()
        t0 = time.perf_counter()
        iters = 20
        for _ in range(iters): dist.all_reduce(t)
        ts.append((time.perf_counter() - t0) / iters * 1e3)
    if rank == 0:
        import numpy as np
        S_arr = np.array(sizes, float); t_arr = np.array(ts)
        b, a = np.polyfit(S_arr, t_arr, 1)  # t = a + b*S
        alpha_us = a / (2 * (N - 1)) * 1e3
        B_GBs = 2 * (N - 1) / N / (b * 1e-3) / 1e9
        for S, dt in zip(sizes, ts):
            print(f'  {S//1024:>6d} KB -> {dt:.3f} ms/op')
        print(f'拟合: alpha ≈ {alpha_us:.1f} us/op, 有效带宽 ≈ {B_GBs:.2f} GB/s')
        print('（gloo over TCP 的数量级；GPU 上换 NCCL + NVLink/IB 后 B 大 1-2 个数量级、alpha 小得多）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(bench, args=(2,), nprocs=2, join=True)


In [ ]:
!python3 ddp_bench.py

## 8. DDP 没解决什么：显存墙原样还在

回到第 1 节的账：DDP 里每个 rank 都存**全量**参数 + 全量梯度 + 全量优化器状态，16 字节/参数一分不少。70B 模型单卡显存需求仍是 1.1TB —— DDP 只是让你用 $ N $ 张卡把计算摊开，**没有把任何一份副本变小**。

所以纯 DDP 的可用范围是"单卡装得下的模型、多卡加速"。模型本身装不下时，需要把参数/梯度/优化器状态**切开存** —— 这就是 ZeRO 的三级分片和 PyTorch 的 FSDP，下一篇的内容。两个问题的分工先记清：

| 问题 | 解法 | 在哪解决 |
|---|---|---|
| 梯度同步贵 | ring all-reduce + 分桶 + 重叠 | DDP（本篇） |
| 单卡装不下 | 参数/梯度/优化器状态分片 | ZeRO / FSDP（下一篇） |


## 9. Debug 演练：两个可跑的 bug 现场

调试清单（下一节）每条都挂在第 6 节的两个前提上，其中最要紧的两条这里跑给你看。

In [ ]:
%%writefile ddp_bug_reinit.py
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

def run_bug(rank, world_size):
    """mp.spawn 入口（rank / world_size 含义同第 6 节）：包 DDP 之后又逐 rank 重初始化，前提 1 当场破功。"""
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29511'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    torch.manual_seed(0)
    model = DDP(torch.nn.Linear(10, 1))          # 构造时已从 rank 0 广播，本来是一致的
    # ← BUG：包完 DDP 又按各 rank 本地种子重新初始化
    torch.manual_seed(100 + rank)
    with torch.no_grad():
        for pp in model.parameters():
            pp.normal_()
    params = torch.cat([p.data.view(-1) for p in model.parameters()])
    gathered = [torch.zeros_like(params) for _ in range(world_size)]
    dist.all_gather(gathered, params)
    if rank == 0:
        d = (gathered[0] - gathered[1]).abs().max().item()
        print(f'BUG 1 症状: 两 rank 参数最大差 = {d:.3f}（≠0；第 6 节正确版是 0）—— 此后每步平均的都是不同模型的梯度')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run_bug, args=(2,), nprocs=2, join=True)

In [ ]:
!python3 ddp_bug_reinit.py

**修复 1**：所有初始化必须在包 DDP **之前**完成（同种子，或先建后由 rank 0 广播），第 6 节的 smoke test 就是正确姿势。

**为什么有效**：DDP 的构造广播只保证"包的那一刻"一致，之后谁再动参数谁负责。重初始化这种操作单机毫无感觉，多机直接把前提 1 拆了——而且症状不在 loss 上先出现，是在参数一致性检查上：所以 smoke test 的固定动作是训一步后 all_gather 比参数，最大差必须为 0。

In [ ]:
%%writefile ddp_fix_reinit.py
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

def run_fix(rank, world_size):
    """mp.spawn 入口：修复版 —— 初始化全部在包 DDP 之前完成，参数一致性恢复。"""
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29514'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    torch.manual_seed(0)                       # 全 rank 同种子，先把模型建一致
    base = torch.nn.Linear(10, 1)
    model = DDP(base)                          # 再包 DDP（构造广播是第二道保险）
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    torch.manual_seed(1234 + rank)             # 数据按 rank 分片（参数种子已在上面定死，这里只动数据）
    x = torch.randn(32, 10); y = torch.randn(32, 1)
    opt.zero_grad()
    loss = ((model(x) - y) ** 2).mean()
    loss.backward(); opt.step()                # 各 rank 用不同数据训一步
    params = torch.cat([p.data.view(-1) for p in model.parameters()])
    gathered = [torch.zeros_like(params) for _ in range(world_size)]
    dist.all_gather(gathered, params)
    if rank == 0:
        d = (gathered[0] - gathered[1]).abs().max().item()
        print(f'FIX 1: 训了一步后两 rank 参数最大差 = {d:.2e}（=0 = 修好；BUG 1 是 2.93）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run_fix, args=(2,), nprocs=2, join=True)

In [ ]:
!python3 ddp_fix_reinit.py

In [ ]:
%%writefile ddp_bug_bn.py
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

def run_bn(rank, world_size):
    """mp.spawn 入口：普通 BN 在 DDP 下用本地 batch 统计量，同一个样本在不同 rank 被归一化成不同的值。"""
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29512'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)

    probe = torch.tensor([[1.0, 2.0, 3.0, 4.0]])            # 同一个样本，放进两个 rank 的 batch
    torch.manual_seed(rank)
    fill = torch.randn(63, 4) * 0.1 + (10.0 if rank == 1 else 0.0)   # 两 rank 的数据分布不同
    batch = torch.cat([probe, fill], 0)

    def probe_out(module):
        module.train()
        out = module(batch)[0]
        g = [torch.zeros_like(out) for _ in range(world_size)]
        dist.all_gather(g, out)
        return (g[0] - g[1]).abs().max().item()

    d_local = probe_out(torch.nn.BatchNorm1d(4, affine=False))          # DDP 默认的本地 BN 行为
    d_ln = probe_out(torch.nn.LayerNorm(4, elementwise_affine=False))   # 不依赖 batch 统计的归一化
    if rank == 0:
        print(f'BUG 2 症状: 同一 probe 在两 rank 的输出最大差：本地 BN = {d_local:.3f}（≠0），LayerNorm = {d_ln:.2e}（=0，与 batch 构成无关）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run_bn, args=(2,), nprocs=2, join=True)

In [ ]:
!python3 ddp_bug_bn.py

**修复 2**：两个方向选一个：GPU 训练的标准解是把 BN 换成 SyncBatchNorm（`torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)` 一行转换，它在 forward 里先 all-reduce 对齐各 rank 的统计量，需要 GPU 运行时）；或者干脆换 LayerNorm 这类不依赖 batch 统计的归一化（上面的演示就是它，CPU 上就能跑）。

**为什么有效**：普通 BN 在 train 模式下用的是"当前 rank 这批数据"的均值方差——各 rank 数据分布不同，同一个样本被按不同标准归一化，相当于每个 rank 在训一个略有不同的模型，梯度平均时互相拖拽。把统计量对齐（SyncBN）或把 batch 统计整个拿掉（LN），语义才和单机一致。这也是清单里"loss 不降但单卡正常"的头号嫌疑。

## 10. 调试速查（每条都挂在第 6 节的两个前提上）

可跑的两条在第 9 节：包 DDP 后重初始化（BUG 1）、本地 BN 统计量（BUG 2）。其余记住处置：

1. **Hang 住不动** —— 某 rank 没进这次 collective（前提 2 破了），定位用 `TORCH_DISTRIBUTED_DEBUG=DETAIL`，GPU 上加 `NCCL_DEBUG=INFO`；根治是保证每 rank 每步 collective 次数一致。
2. **loss 不降但单卡正常** —— 先查 BN（BUG 2），再查梯度平均后 lr 是否要按 N 放大（线性缩放规则）。
3. **`find_unused_parameters=True`** —— 每步多遍历一次计算图，真有条件分支才开。
4. **混合精度静默发散** —— `GradScaler` 在 DDP 外层，`unscale_` 在 all-reduce 之后、clip 之前，顺序别换。
5. **起进程** —— 生产用 `torchrun --nproc_per_node=2 day-01-ddp-minimal.py`；在 notebook 里跑 spawn 演示必须先落成 `.py` 脚本再运行（直接在 cell 里定义函数再 `mp.spawn`，子进程 import 不到 `__main__` 里的函数，必挂 `ProcessExitedException` —— 本篇第 6 节的运行方式说明）。

## 小结：因果链回顾

单卡两堵墙（16 字节/参数的显存墙、$ 6P $ FLOPs/token 的时间墙）→ 数据并行切 batch（数学上等价于大 batch，代价是每步同步一份参数量级的梯度）→ 参数服务器死于 $ 2NS $ 的单点带宽 → ring all-reduce 把它降到 $ 2(N-1)S/N $ 并达到带宽下界 → 小梯度的延迟问题由分桶解决，通信由与 backward 重叠藏起来 → DDP 的两个一致性前提（初始化一致、collective 次数一致）解释了几乎所有调试症状 → 但显存墙原样还在，留给 ZeRO/FSDP。

面试映射：这条链对应的正是并行基础那一轮的问法 —— 不会只让你写公式，而是"70B 模型 1024 卡，DDP 能不能训、瓶颈在哪"这种账。能把每环的账算出来，就是这一轮的满分形态。

**下一篇**：ZeRO / FSDP —— 把 16 字节/参数按 $ N $ 切开，每级各付多少通信代价。
